In [1]:
import os, random, logging, sys
import numpy as np
import pandas as pd
from collections import Counter

import torch
from transformers import (
    AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, default_data_collator, set_seed
)
from datasets import Dataset, DatasetDict
import evaluate

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

In [ ]:
train_file = "blp25/dataset/1B/train.tsv"
validation_file = "blp25/dataset/1B/dev.tsv"
test_file = "blp25/dataset/1B/dev_test.tsv"

In [3]:
# Label mapping
l2id = {'None': 0, 'Society': 1, 'Organization': 2, 'Community': 3, 'Individual': 4}
id2l = {v:k for k,v in l2id.items()}
label_list = list(l2id.values())
num_labels = len(l2id)

In [4]:
# Load datasets
def load_datasets():
    train_df = pd.read_csv(train_file, sep="\t")
    train_df['label'] = train_df['label'].map(l2id).fillna(0).astype(int)
    validation_df = pd.read_csv(validation_file, sep="\t")
    validation_df['label'] = validation_df['label'].map(l2id).fillna(0).astype(int)
    test_df = pd.read_csv(test_file, sep="\t")
    
    return DatasetDict({
        "train": Dataset.from_pandas(train_df),
        "validation": Dataset.from_pandas(validation_df),
        "test": Dataset.from_pandas(test_df),
    })

In [5]:
def train_and_predict(model_name, epochs, output_dir="./outputs"):
    raw_datasets = load_datasets()
    config = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

    def preprocess(examples):
        return tokenizer(examples["text"], max_length=128, padding="max_length", truncation=True)

    raw_datasets = raw_datasets.map(preprocess, batched=True)

    train_dataset = raw_datasets["train"].remove_columns("id")
    eval_dataset = raw_datasets["validation"].remove_columns("id")
    predict_dataset = raw_datasets["test"]

    training_args = TrainingArguments(
        learning_rate=2e-5,
        num_train_epochs=1,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        output_dir=output_dir,
        overwrite_output_dir=True,
        save_strategy="no",
        report_to=None,
        load_best_model_at_end=True,
        greater_is_better=True,
        fp16=True,
        logging_steps=100,
    )

    metric = evaluate.load("accuracy")
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=1)
        return {"accuracy": (preds == p.label_ids).mean().item()}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer,
        data_collator=default_data_collator,
    )
    trainer.train()

    ids = predict_dataset["id"]
    pred_logits = trainer.predict(predict_dataset.remove_columns("id")).predictions
    probs = torch.softmax(torch.tensor(pred_logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)

    out_path = os.path.join(output_dir, "subtask_1B.tsv")
    pd.DataFrame({
        "id": ids,
        "label": [id2l[i] for i in preds],
        "model": model_name
    }).to_csv(out_path, sep="\t", index=False)

    np.save(os.path.join(output_dir, "probs.npy"), probs)
    return ids, preds, probs


In [6]:

# Run all models
models = [
    "google/muril-large-cased",
    'ai4bharat/IndicBERTv2-MLM-only',
    "csebuetnlp/banglabert_large"
]
all_probs = []
for m in models:
    _, preds, probs = train_and_predict(m, output_dir=m.replace("/","__"), epochs=1)
    all_probs.append(probs)

ids = pd.read_csv(test_file, sep="\t",)["id"]

# Hard vote
hard_labels = []
for row in zip(*[p.argmax(1) for p in all_probs]):
    hard_labels.append(Counter(row).most_common(1)[0][0])
hard_labels = [id2l[i] for i in hard_labels]
pd.DataFrame({"id": ids, "label": hard_labels, "model": "hv_ensemble_muril_bert_indic_1"}).to_csv("submission_hard.tsv", sep="\t", index=False)

# Soft vote
soft_probs = np.mean(all_probs, axis=0)
soft_preds = [id2l[i] for i in soft_probs.argmax(1)]
pd.DataFrame({"id": ids, "label": soft_preds, "model": "sv_ensemble_muril_bert_indic_1"}).to_csv("submission_soft.tsv", sep="\t", index=False)

# Weighted vote
weights = [0.4, 0.35, 0.25]  # tuneable
w_probs = np.tensordot(all_probs, weights, axes=(0,0))
w_preds = [id2l[i] for i in w_probs.argmax(1)]
pd.DataFrame({"id": ids, "label": w_preds, "model": "wv_ensemble_muril_bert_indic_1"}).to_csv("submission_weighted.tsv", sep="\t", index=False)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-large-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/35522 [00:00<?, ? examples/s]

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]

/tmp/ipykernel_136113/4062797956.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/nafi/dev/shared-task/blp25/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


...

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]

Map:   0%|          | 0/2512 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

/tmp/ipykernel_136113/4062797956.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/nafi/dev/shared-task/blp25/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `ElectraSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Step,Training Loss
100,1.137600
200,1.024600
300,0.916300
400,0.909300
500,0.878300
600,0.817000
700,0.831200
800,0.797100
900,0.799300
1000,0.790900
